<a href="https://colab.research.google.com/github/f-ai0/ds-training/blob/main/week7/Day5_Checkpoints_Final_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch torchvision

# Task 5.1 — Save and Load a Checkpoint

In [3]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('week7', exist_ok=True)

# Quick re-train of Day 2's model so we have something to checkpoint
df = pd.read_excel('Practice_Dataset.xlsx')
features = ['punch_count', 'hours_worked', 'satisfaction_score', 'monthly_salary']
X = SimpleImputer(strategy='median').fit_transform(df[features])
y = df['is_absent'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)

class AbsenceClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, x):
        return self.net(x)

model = AbsenceClassifier(input_dim=X_train_t.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(50):
    model.train()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Save a FULL checkpoint - not just weights.
# Saving only model.state_dict() loses optimizer momentum/Adam moments,
# so resuming training later would restart optimization from scratch.
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item(),
}
torch.save(checkpoint, 'week7/checkpoint.pth')
print(f'Saved checkpoint at epoch {epoch}, loss={loss.item():.4f}')

Saved checkpoint at epoch 49, loss=0.4651


In [4]:
# Load the checkpoint into a FRESH model to verify it works
loaded_checkpoint = torch.load('week7/checkpoint.pth', map_location=device)

new_model = AbsenceClassifier(input_dim=X_train_t.shape[1]).to(device)
new_optimizer = torch.optim.Adam(new_model.parameters(), lr=0.001)

new_model.load_state_dict(loaded_checkpoint['model_state_dict'])
new_optimizer.load_state_dict(loaded_checkpoint['optimizer_state_dict'])

print(f"Loaded checkpoint from epoch {loaded_checkpoint['epoch']}, ",
      f"loss={loaded_checkpoint['loss']:.4f}")

# Verify: predictions from the loaded model should match the original exactly
new_model.eval()
model.eval()
with torch.no_grad():
    original_out = model(X_train_t[:5])
    loaded_out = new_model(X_train_t[:5])

print(f'Outputs match: {torch.allclose(original_out, loaded_out)}')

Loaded checkpoint from epoch 49,  loss=0.4651
Outputs match: True


# Task 5.2 — Resume Training from a Checkpoint

In [5]:
# Resume training - loss should continue smoothly from where it left off,
# not spike back up, because the Adam optimizer state was restored too.
for epoch in range(loaded_checkpoint['epoch'] + 1, loaded_checkpoint['epoch'] + 21):
    new_model.train()
    outputs = new_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    new_optimizer.zero_grad()
    loss.backward()
    new_optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}')

Epoch 55: loss=0.4403
Epoch 60: loss=0.4067
Epoch 65: loss=0.3862
Epoch 70: loss=0.3610


# Task 5.3 — Save the Final Model

In [6]:
# For deployment, only weights are needed - no optimizer state
torch.save(new_model.state_dict(), 'week7/final_model_weights.pth')
print('Final model weights saved to week7/final_model_weights.pth')

Final model weights saved to week7/final_model_weights.pth


# Task 5.4 — Honest PyTorch vs TensorFlow Comparison

In [8]:
comparison_report = """
PyTorch vs TensorFlow - Week 7 Comparison
==========================================

1) Tabular MLP (is_absent classification)
   TensorFlow (Week 6): Accuracy=1.000, F1=1.000, AUC=1.000
   PyTorch    (Week 7): Accuracy=0.903, F1=0.000, AUC=1.000
   -> Both models learned to separate the classes equally well (AUC=1.0),
      but PyTorch's F1 collapsed to 0 at the default 0.5 threshold because
      the test set was imbalanced (7/72 positive). This is a threshold
      issue, not a learning-capacity issue - fixable with a lower
      threshold or class weighting (see Day 2 notes).

2) CNN on Fashion-MNIST
   TensorFlow (Week 6): Test accuracy = 0.8965 (10 epochs)
   PyTorch    (Week 7): Test accuracy = 0.8960 (5 epochs)
   -> Nearly identical final accuracy. PyTorch reached it in half the
      epochs, likely due to similar architecture choices but possibly
      a slightly more effective optimizer/learning-rate combination.

3) Transfer Learning (ResNet18 on CIFAR-10 subset)
   No direct Week 6 equivalent - this was new in Week 7.
   Demonstrates that fine-tuning a pretrained model on ~5,000 images
   reaches strong accuracy in just 3 epochs, versus the much larger
   data and longer training needed for the from-scratch CNN above.

Overall takeaway:
Both frameworks reach comparable model quality on the same tasks - the
real differences were in code style (PyTorch's explicit training loop
vs TensorFlow's .fit()) and in details like decision thresholds, which
matter more than the framework choice itself on imbalanced data.
"""

print(comparison_report)

with open('week7/comparison_report.txt', 'w') as f:
    f.write(comparison_report)
print('Saved to week7/comparison_report.txt')


PyTorch vs TensorFlow - Week 7 Comparison

1) Tabular MLP (is_absent classification)
   TensorFlow (Week 6): Accuracy=1.000, F1=1.000, AUC=1.000
   PyTorch    (Week 7): Accuracy=0.903, F1=0.000, AUC=1.000
   -> Both models learned to separate the classes equally well (AUC=1.0),
      but PyTorch's F1 collapsed to 0 at the default 0.5 threshold because
      the test set was imbalanced (7/72 positive). This is a threshold
      issue, not a learning-capacity issue - fixable with a lower
      threshold or class weighting (see Day 2 notes).

2) CNN on Fashion-MNIST
   TensorFlow (Week 6): Test accuracy = 0.8965 (10 epochs)
   PyTorch    (Week 7): Test accuracy = 0.8960 (5 epochs)
   -> Nearly identical final accuracy. PyTorch reached it in half the
      epochs, likely due to similar architecture choices but possibly
      a slightly more effective optimizer/learning-rate combination.

3) Transfer Learning (ResNet18 on CIFAR-10 subset)
   No direct Week 6 equivalent - this was new in We